Test programtically replaying matches (for use in behavioral cloning)

In [2]:
from pathlib import Path

from ssf2_rl.env.gym_env import SSF2Env
from ssf2_rl.game.controls import describe_mask


Loading 2026-08-29 11.02 AM - Versus - P1 (Marth) vs CPU Lvl 9 (Wario).ssfrec ...
Collected 357 frames (frame 1..357)


In [ ]:

# Load a human-recorded .ssfrec replay and collect per-frame states.
# The game plays the replay back; every frame is buffered game-side and
# bulk-transferred to Python at the end (no per-frame socket round-trips).
# replay_path = next(Path("data/toy").glob("*.ssfrec"))
replay_path = Path("data/toy/2026-08-29 11.02 AM - Versus - P1 (Marth) vs CPU Lvl 9 (Wario).ssfrec")
print(f"Loading {replay_path.name} ...")

env = SSF2Env(step_timeout=10.0)
try:
    episode = env.replay_ssfrec(str(replay_path), collect=True)
finally:
    env.close()

print(f"Collected {len(episode)} frames "
      f"(frame {episode.frames[0]['frame']}..{episode.frames[-1]['frame']})")


In [5]:
# Extract (observation, action) pairs for behavioral cloning.
obs, actions = episode.to_bc_dataset(player_id=1)
print(f"BC dataset: obs {obs.shape}, actions {actions.shape}")

BC dataset: obs (357, 38), actions (357,)


In [ ]:
# Sanity-check the recorded human inputs.
from collections import Counter
nonzero = [a for a in actions if a != 0]
print(f"Frames with input: {len(nonzero)}/{len(actions)} "
      f"({100 * len(nonzero) / len(actions):.1f}%)")
for mask, count in Counter(nonzero).most_common(5):
    print(f"  {describe_mask(int(mask))}: {count} frames")

Frames with input: 2586/3008 (86.0%)
  RIGHT|DASH: 574 frames
  LEFT|DASH: 454 frames
  DASH: 233 frames
  SHIELD: 106 frames
  DOWN|DASH: 92 frames


In [5]:
#obs, actions
print(obs)

[[-3.2962501e-01  1.0937500e-01  0.0000000e+00 ...  1.0000000e+00
   9.9999997e-06 -1.0000000e+00]
 [-3.2962501e-01  1.0937500e-01  0.0000000e+00 ...  1.0000000e+00
   1.9999999e-05 -1.0000000e+00]
 [-3.2962501e-01  1.0937500e-01  0.0000000e+00 ...  1.0000000e+00
   2.9999999e-05 -1.0000000e+00]
 ...
 [ 5.9650004e-01  2.6162499e-01 -2.6885075e-02 ... -1.0000000e+00
   3.0060001e-02 -1.0000000e+00]
 [ 5.9650004e-01  2.6400000e-01  0.0000000e+00 ... -1.0000000e+00
   3.0069999e-02 -1.0000000e+00]
 [ 5.9650004e-01  2.6637501e-01  0.0000000e+00 ... -1.0000000e+00
   3.0080000e-02 -1.0000000e+00]]


In [ ]:
# Save the collected episode for offline training
episode.save("data/collected_episode.json")
print("Saved to data/collected_episode.json")

# Episodes can be reloaded without the game:
from ssf2_rl.data.episode import Episode
loaded = Episode.load("data/collected_episode.json")
obs2, actions2 = loaded.to_bc_dataset(player_id=1)
assert obs2.shape == obs.shape
print("Reload OK — training can happen fully offline")

Saved to data/collected_episode.json
Reload OK — training can happen fully offline


^ Normal/native speed (30fps) rendering. Testing faster replay rendering

In [ ]:
# Works

from time import perf_counter

replay_path = Path("data/toy/2026-08-27 11.38 AM - Versus - P1 (Marth) vs CPU Lvl 9 (Samus).ssfrec")
started = perf_counter()

env = SSF2Env(step_timeout=10.0)
try:
    fast_episode = env.replay_ssfrec(
        str(replay_path), collect=True, speed="fast", batch_frames=256
    )
finally:
    env.close()




elapsed = perf_counter() - started
fast_obs, fast_actions = fast_episode.to_bc_dataset(player_id=1)
assert fast_obs.shape == (len(fast_episode), 38)
assert fast_actions.dtype.name == "int32"
assert all(
    next_frame["frame"] == frame["frame"] + 1
    for frame, next_frame in zip(fast_episode.frames, fast_episode.frames[1:])
)
print(
    f"Fast collection: {len(fast_episode)} frames in {elapsed:.3f}s; "
    f"obs={fast_obs.shape}, actions={fast_actions.shape}"
)

Fast collection: 3008 frames in 3.290s; obs=(3008, 38), actions=(3008,)
